# Modelado

## 1. Preparación de los datos

Se separan las variables de entrada (`X`) de la variable objetivo (`y`) y se divide el dataset en entrenamiento y test. La división se hace antes de cualquier transformación para evitar fuga de datos, y es estratificada para mantener la misma proporción de fallos en ambos conjuntos ya que el dataset está desbalanceado.

In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split

# Guardamos la versión de scikit-learn: la API de despliegue tendrá que usar la misma
print(f"Versión de scikit-learn: {sklearn.__version__}")

# Cargamos el dataset limpio guardado al final de la fase de Data Cleaning
df_clean = pd.read_csv("../data/ai4i2020_clean.csv")
print(f"Dimensiones del dataset limpio: {df_clean.shape[0]} filas y {df_clean.shape[1]} columnas.")

Versión de scikit-learn: 1.8.0
Dimensiones del dataset limpio: 9981 filas y 7 columnas.


In [2]:
# X: variables de entrada. y: variable objetivo (lo que queremos predecir)
X = df_clean.drop(columns=["Machine failure"])
y = df_clean["Machine failure"]

# División en train (80 %) y test (20 %)
# - stratify=y mantiene la misma proporción de fallos en ambos conjuntos
# - random_state fija la semilla para que el resultado sea reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape[0]} filas | Test: {X_test.shape[0]} filas")

# Comprobamos que la proporción de fallos se mantiene en los dos conjuntos
print("\nPorcentaje de fallos en train:", round(y_train.mean() * 100, 2), "%")
print("Porcentaje de fallos en test:", round(y_test.mean() * 100, 2), "%")

Train: 7984 filas | Test: 1997 filas

Porcentaje de fallos en train: 3.38 %
Porcentaje de fallos en test: 3.41 %


### Conclusión división en train y test

- El conjunto de entrenamiento tiene 7.984 filas y el de test 1.997 (aproximadamente un 80 % y un 20 % de las 9.981 filas).
- El porcentaje de fallos es del 3,38 % en train y del 3,41 % en test, muy cercano al 3,39 % del dataset completo. La estratificación ha mantenido la proporción de fallos en ambos conjuntos.
- El conjunto de test se reserva y no se vuelve a usar hasta la evaluación final.

## 2. Preprocesado

Se define el preprocesado de cada tipo de variable: 

`Type` se codifica con one-hot encoding y las variables numéricas se escalan con `StandardScaler`. 

Los outliers no se eliminan, ya que en el análisis exploratorio aparecieron sobre todo en los registros con fallo. El preprocesado se incluirá dentro de un `Pipeline`, de forma que en cada partición de la validación cruzada se ajuste solo con los datos de entrenamiento.

In [3]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7984 entries, 6947 to 7474
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Type                     7984 non-null   object 
 1   Air temperature [K]      7984 non-null   float64
 2   Process temperature [K]  7984 non-null   float64
 3   Rotational speed [rpm]   7984 non-null   int64  
 4   Torque [Nm]              7984 non-null   float64
 5   Tool wear [min]          7984 non-null   int64  
dtypes: float64(3), int64(2), object(1)
memory usage: 436.6+ KB


In [4]:
X_train.describe()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
count,7984.000000,7984.000000,7984.000000,7984.000000,7984.000000
mean,300.003307,310.006162,1539.764905,39.922620,107.653557
std,1.997147,1.478629,181.238533,10.003102,63.656693
min,295.300000,305.700000,1168.000000,3.800000,0.000000
25%,298.300000,308.800000,1423.000000,33.100000,53.000000
50%,300.100000,310.100000,1503.000000,40.000000,108.000000
75%,301.500000,311.100000,1614.000000,46.700000,162.000000
max,304.500000,313.800000,2886.000000,76.600000,253.000000


In [5]:
X_train["Type"].value_counts()

Type
L    4774
M    2411
H     799
Name: count, dtype: int64

In [6]:
features_cat = [
   'Type'
]

features_num = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
]

In [7]:
# Calculamos la asimetría para las features numéricas
X_train[features_num].skew().sort_values(ascending=False).round(2)

Rotational speed [rpm]     2.02
Air temperature [K]        0.12
Tool wear [min]            0.04
Process temperature [K]    0.03
Torque [Nm]               -0.01
dtype: float64

In [8]:
# Features numéricas que no necesitan transformación
features_num_normal = ['Air temperature [K]', 'Tool wear [min]', 'Process temperature [K]', 'Torque [Nm]']

# Features numéricas sesgadas transformación logarítmica
features_num_skew_log = ['Rotational speed [rpm]']

## Pipelines para todos los modelos
Se crean 3 pipelines diferentes para cubrir correctamente todos los tipos de modelos según sus necesidades de preprocesamiento.

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

# ======================================================================
# GRUPO 1: modelos que necesitan escalado
# Regresión logística, KNN y SVM
# Rotational speed: log1p + escalado | resto de numéricas: escalado | Type: one-hot
# ======================================================================
num_pipeline = Pipeline([
    ("scale", StandardScaler()),
])

num_skew_log_pipeline = Pipeline([
    ("log", FunctionTransformer(np.log1p, validate=False)),  # transformación logarítmica
    ("scale", StandardScaler()),                             # escalado
])

cat_pipeline = Pipeline([
    ("one_hot", OneHotEncoder(drop="if_binary", handle_unknown="ignore")),
])

preprocessing_sup_1 = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, features_num_normal),
        ("num_skew_log", num_skew_log_pipeline, features_num_skew_log),
        ("cat", cat_pipeline, features_cat),
    ],
    remainder="drop",
)

# ======================================================================
# GRUPO 2: modelos de árboles sin escalado
# Random Forest, XGBoost y LightGBM
# Las numéricas pasan tal cual | Type: one-hot
# ======================================================================
cat_pipeline_2 = Pipeline([
    ("one_hot", OneHotEncoder(drop="if_binary", handle_unknown="ignore")),
])

preprocessing_sup_2 = ColumnTransformer(
    transformers=[
        ("num", "passthrough", features_num),
        ("cat", cat_pipeline_2, features_cat),
    ],
    remainder="drop",
)

# ======================================================================
# GRUPO 3: CatBoost
# Gestiona las categóricas por sí mismo, así que no se codifican.
# - verbose_feature_names_out=False mantiene los nombres originales de las columnas
# - set_output(transform="pandas") devuelve un DataFrame, que conserva los tipos
#   (números y texto) y permite indicar la categórica por su nombre
# ======================================================================
preprocessing_catboost = ColumnTransformer(
    transformers=[
        ("num", "passthrough", features_num),
        ("cat", "passthrough", features_cat),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# Desbalanceo: class_weight="balanced" da más peso a los errores cometidos con los fallos,
# para que el modelo penalice más no detectarlos.
# XGBoost no tiene class_weight: usa scale_pos_weight = registros sin fallo / registros con fallo.
peso_clases = np.sum(y_train == 0) / np.sum(y_train == 1)

# n_jobs=1 / thread_count=1 en los modelos: el paralelismo lo hace la validación cruzada
# (n_jobs=-1 en cross_validate), así los modelos no compiten por los procesadores.
modelos_base = {
    "Regresión Logística": Pipeline([
        ("preprocesador", preprocessing_sup_1),
        ("modelo", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
    ]),
    "Random Forest": Pipeline([
        ("preprocesador", preprocessing_sup_2),
        ("modelo", RandomForestClassifier(class_weight="balanced", n_jobs=1, random_state=42)),
    ]),
    "XGBoost": Pipeline([
        ("preprocesador", preprocessing_sup_2),
        ("modelo", xgb.XGBClassifier(scale_pos_weight=peso_clases, random_state=42,
                                     eval_metric="logloss", n_jobs=1)),
    ]),
    "LightGBM": Pipeline([
        ("preprocesador", preprocessing_sup_2),
        ("modelo", lgb.LGBMClassifier(class_weight="balanced", random_state=42,
                                      n_jobs=1, verbose=-1)),
    ]),
    "KNN": Pipeline([
        ("preprocesador", preprocessing_sup_1),
        ("modelo", KNeighborsClassifier(n_neighbors=5, n_jobs=1)),  # KNN no admite class_weight
    ]),
    "SVM": Pipeline([
        ("preprocesador", preprocessing_sup_1),
        # Sin probability=True: no hace falta para comparar con validación cruzada y va mucho más rápido
        ("modelo", SVC(class_weight="balanced", random_state=42)),
    ]),
    "CatBoost": Pipeline([
    # Mismo preprocesado que los otros modelos de árboles: Type con one-hot encoding.
    # Así no hace falta declarar cat_features, que da problemas al clonar el modelo
    # dentro de la validación cruzada.
    ("preprocesador", preprocessing_sup_2),
    ("modelo", CatBoostClassifier(
        auto_class_weights="Balanced", thread_count=1, verbose=0, random_state=42)),
]),
}

print("Los 7 modelos base están listos.")

Los 7 modelos base están listos.


In [16]:
import time
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate


def evaluar_modelos_cv(modelos, X, y, cv=None, metricas=None, orden="pr_auc", n_jobs=-1):
    """
    Evalúa varios modelos con validación cruzada y devuelve una tabla resumen.

    Parámetros:
    - modelos: diccionario {nombre: Pipeline} con los modelos a comparar.
    - X, y: datos de ENTRENAMIENTO (nunca el conjunto de test).
    - cv: validación cruzada a usar. Por defecto, StratifiedKFold de 5 particiones.
    - metricas: diccionario {nombre_corto: métrica de scikit-learn}.
    - orden: métrica por la que se ordena la tabla (por defecto, PR-AUC).
    - n_jobs: procesadores para la validación cruzada (-1 = todos).

    Devuelve un DataFrame con la media y la desviación de cada métrica entre particiones.
    """
    # Validación cruzada estratificada: mantiene la proporción de fallos en cada partición
    if cv is None:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Métricas por defecto (PR-AUC es la principal por el desbalanceo de clases)
    if metricas is None:
        metricas = {
            "pr_auc": "average_precision",
            "recall": "recall",
            "precision": "precision",
            "f1": "f1",
            "roc_auc": "roc_auc",
        }

    filas = []

    for nombre, pipe in modelos.items():
        print(f"Evaluando: {nombre}...", end=" ")
        inicio = time.time()

        # cross_validate clona el pipeline en cada partición, así que el preprocesado
        # se ajusta solo con los datos de entrenamiento de esa partición (sin fuga de datos).
        # error_score="raise": si un modelo falla, se muestra el error en vez de rellenar con NaN.
        scores = cross_validate(
            pipe, X, y,
            cv=cv, scoring=metricas,
            n_jobs=n_jobs, error_score="raise",
        )

        # Guardamos la media y la desviación entre particiones de cada métrica
        fila = {"modelo": nombre}
        for m in metricas:
            fila[m] = scores[f"test_{m}"].mean()
            fila[f"{m}_std"] = scores[f"test_{m}"].std()
        fila["tiempo_fit_s"] = scores["fit_time"].mean()
        filas.append(fila)

        print(f"listo ({time.time() - inicio:.0f} s)")

    # Tabla ordenada de mejor a peor según la métrica elegida
    tabla = pd.DataFrame(filas).sort_values(orden, ascending=False).reset_index(drop=True)
    return tabla.round(3)

In [17]:
tabla_cv = evaluar_modelos_cv(modelos_base, X_train, y_train)
display(tabla_cv)

Evaluando: Regresión Logística... listo (2 s)
Evaluando: Random Forest... listo (2 s)
Evaluando: XGBoost... listo (1 s)
Evaluando: LightGBM... listo (2 s)
Evaluando: KNN... listo (1 s)
Evaluando: SVM... listo (1 s)
Evaluando: CatBoost... listo (4 s)


,modelo,pr_auc,pr_auc_std,recall,recall_std,precision,precision_std,f1,f1_std,roc_auc,roc_auc_std,tiempo_fit_s
0,LightGBM,0.775,0.029,0.763,0.055,0.671,0.045,0.712,0.030,0.971,0.003,0.116
1,XGBoost,0.774,0.041,0.719,0.072,0.718,0.040,0.715,0.025,0.971,0.006,0.090
2,CatBoost,0.767,0.026,0.774,0.070,0.578,0.041,0.658,0.022,0.968,0.008,3.874
3,Random Forest,0.725,0.046,0.344,0.009,0.891,0.059,0.496,0.010,0.961,0.007,0.418
4,SVM,0.614,0.052,0.874,0.044,0.272,0.023,0.414,0.024,0.960,0.006,0.283
5,KNN,0.431,0.028,0.248,0.048,0.803,0.072,0.375,0.050,0.840,0.046,0.018
6,Regresión Logística,0.361,0.050,0.781,0.041,0.120,0.006,0.208,0.010,0.879,0.015,0.019


### Criterio de negocio y métricas

Para este problema, una parada inesperada de la máquina es más costosa que un mantenimiento preventivo innecesario. Por tanto, se prioriza el **recall** (detectar los fallos reales) frente a la precision (evitar falsas alarmas).

- La **PR-AUC** se mantiene como métrica para comparar modelos y optimizar hiperparámetros, porque no depende del umbral de decisión y es adecuada con clases muy desbalanceadas.
- El **recall** se ajustará después, mediante el umbral de decisión del modelo elegido, buscando detectar el mayor número de fallos posible con un número razonable de falsas alarmas.